<a href="https://colab.research.google.com/github/ryancookd/Predicting-Heart-Disease/blob/main/Predicting_Heart_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e2_path = kagglehub.competition_download('playground-series-s6e2')

print('Data source import complete.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

In [ ]:
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")
sample_sub = pd.read_csv("/kaggle/input/playground-series-s6e2/sample_submission.csv")

def clean_cols(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    return df

train = clean_cols(train)
test = clean_cols(test)
sample_sub = clean_cols(sample_sub)

print(train.shape, test.shape)
train.head()

In [ ]:
train.columns

In [ ]:
TARGET = "Heart_Disease"
ID_COL = "id"

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])

print(X.shape, y.shape, X_test.shape)
print("Target distribution:\n", y.value_counts(dropna=False))

In [ ]:
def add_features(df):
    df = df.copy()

    # Binning
    df["Age_bin"] = pd.cut(df["Age"], bins=5, labels=False).astype("int16")

    # Ratios / nonlinearities
    df["Chol_Age_ratio"] = df["Cholesterol"] / (df["Age"] + 1.0)
    df["ST_squared"] = df["ST_depression"] ** 2

    # Interaction
    df["Age_MaxHR"] = df["Age"] * df["Max_HR"]

    return df

X = add_features(X)
X_test = add_features(X_test)

X.head()

In [ ]:
cat_cols = [
    "Sex",
    "Chest_pain_type",
    "FBS_over_120",
    "EKG_results",
    "Exercise_angina",
    "Slope_of_ST",
    "Number_of_vessels_fluro",
    "Thallium",
]

# Only keep ones that actually exist (safety)
cat_cols = [c for c in cat_cols if c in X.columns]

for col in cat_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print("Categorical columns used:", cat_cols)

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(X), dtype=np.float64)
test_preds = np.zeros(len(X_test), dtype=np.float64)

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.015,
    "num_leaves": 150,
    "max_depth": -1,
    "min_child_samples": 30,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 5,
    "reg_alpha": 1.0,
    "reg_lambda": 1.0,
    "n_estimators": 12000,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        categorical_feature=cat_cols if len(cat_cols) else "auto",
        callbacks=[lgb.early_stopping(300), lgb.log_evaluation(300)],
    )

    oof_preds[va_idx] = model.predict_proba(X_va)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS

oof_auc = roc_auc_score(y, oof_preds)
print(f"OOF AUC Score: {oof_auc:.5f}")

In [ ]:
auc = roc_auc_score(y, oof_preds)
print(f"\nOOF AUC Score: {auc:.5f}")

In [ ]:
submission = sample_sub.copy()

# sample_submission was cleaned too, so target column should match:
# typically it will be Heart_Disease after cleaning
if TARGET not in submission.columns:
    # fallback: use the last column
    pred_col = [c for c in submission.columns if c != ID_COL][0]
else:
    pred_col = TARGET

submission[pred_col] = test_preds
submission.to_csv("submission.csv", index=False)

submission.head()